In [1]:
import pandas as pd
import json
import numpy as np
import os
import urllib.parse
import time
import requests

import sodapy
from sodapy import Socrata
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

from datetime import datetime, timedelta
from pathlib import Path
from datetime import date, timedelta

In [2]:
from dotenv import load_dotenv


In [3]:
# details about the client, token and dataset:
agency = "datacatalog.cookcountyil.gov"
dataset = "cjeq-bs86" # this is the shortlink!
token = os.getenv("COOK_CO_APPTOKEN")

In [4]:

# set time
rightnow = datetime.now().strftime('%Y-%m-%d %I:%M:%S %p')
today = datetime.today().strftime('%Y-%m-%d')
yesterday = datetime.today() - timedelta(1)
yesterday = yesterday.strftime('%Y-%m-%d')
startdate = '2026-10-01'

In [5]:

# set queries, using SoQL. (vom)
today_query = f"SELECT * WHERE cold_related = 'true' AND death_date BETWEEN '2026-10-01' AND '{today}'"
yesterday_query = f"SELECT * WHERE cold_related = 'true' AND death_date BETWEEN '2026-10-01' AND '{yesterday}'"
all_query = f"SELECT * WHERE cold_related = 'true'"



In [6]:
# set the client and authenticate
client = Socrata(agency, 
                 token, 
                 timeout=60)

# get metadata related to the dataset
meta = []
meta = client.get_metadata(dataset)


In [7]:
# pull cold-related deaths from socrata into a dataframe
today_download = client.get(dataset, query=today_query, exclude_system_fields=False)
yesterday_download = client.get(dataset, query=yesterday_query, exclude_system_fields=False)
all_download = client.get(dataset, query=all_query, exclude_system_fields=False)

# # write an if...
# if today_download == NULL

# to dataframe
today_df = pd.DataFrame(today_download)
yesterday_df = pd.DataFrame(yesterday_download)
all_df = pd.DataFrame(all_download)

In [8]:
today_df.columns

Index(['casenumber', 'incident_date', 'death_date', 'age', 'gender', 'race',
       'latino', 'manner', 'primarycause', 'primarycause_linea', 'gunrelated',
       'opioids', 'cold_related', 'heat_related', 'transportation_related',
       'commissioner_district', 'incident_street', 'incident_city',
       'incident_zip', 'longitude', 'latitude', 'location', 'residence_city',
       'residence_zip', 'objectid', 'covid_related', ':id', ':created_at',
       ':updated_at'],
      dtype='str')

In [9]:
yesterday_df.columns

Index(['casenumber', 'incident_date', 'death_date', 'age', 'gender', 'race',
       'latino', 'manner', 'primarycause', 'primarycause_linea', 'gunrelated',
       'opioids', 'cold_related', 'heat_related', 'transportation_related',
       'commissioner_district', 'incident_street', 'incident_city',
       'incident_zip', 'longitude', 'latitude', 'location', 'residence_city',
       'residence_zip', 'objectid', 'covid_related', ':id', ':created_at',
       ':updated_at'],
      dtype='str')

In [11]:
all_df.columns

Index(['casenumber', 'incident_date', 'death_date', 'age', 'gender', 'race',
       'latino', 'manner', 'primarycause', 'primarycause_linea', 'gunrelated',
       'opioids', 'cold_related', 'heat_related', 'transportation_related',
       'incident_street', 'incident_city', 'incident_zip', 'objectid',
       'covid_related', ':id', ':created_at', ':updated_at', 'secondarycause',
       'commissioner_district', 'longitude', 'latitude', 'location',
       'residence_city', 'residence_zip', 'chi_ward', 'chi_commarea',
       'primarycause_lineb', 'transportation_status',
       'transportation_related_type', 'primarycause_linec'],
      dtype='str')

In [12]:
all_df[(all_df['death_date'] > '2026-10-01') & (all_df['death_date'] <= '2026-11-21')]

,casenumber,incident_date,death_date,age,gender,race,latino,manner,primarycause,primarycause_linea,...,latitude,location,residence_city,residence_zip,chi_ward,chi_commarea,primarycause_lineb,transportation_status,transportation_related_type,primarycause_linec
705,ME2026-06079,2026-11-13T07:12:00.000,2026-11-13T06:22:00.000,58,Male,White,False,ACCIDENT,ENVIRONMENTAL COLD EXPOSURE,ENVIRONMENTAL COLD EXPOSURE,...,41.715315,"{'latitude': '41.715315', 'longitude': '-87.69...",Palos Park,60464,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
today_df = all_df[(all_df['death_date'] >= '2026-10-01') & (all_df['death_date'] <= today)]
today_df

,casenumber,incident_date,death_date,age,gender,race,latino,manner,primarycause,primarycause_linea,...,latitude,location,residence_city,residence_zip,chi_ward,chi_commarea,primarycause_lineb,transportation_status,transportation_related_type,primarycause_linec
705,ME2026-06079,2026-11-13T07:12:00.000,2026-11-13T06:22:00.000,58,Male,White,False,ACCIDENT,ENVIRONMENTAL COLD EXPOSURE,ENVIRONMENTAL COLD EXPOSURE,...,41.715315,"{'latitude': '41.715315', 'longitude': '-87.69...",Palos Park,60464,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
chilen = all_df[all_df['incident_city']=='CHICAGO']

In [17]:
len(chilen)

491

In [18]:
len(all_df)

706